# 03 - Model Comparison

Builds the assignment's required outputs entirely from what `02_experiments.
ipynb` already wrote to `results/` - the `ExperimentTracker` log and the
saved per-model/per-square predictions - so nothing here recomputes a
prediction or a metric; this notebook is presentation and interpretation of
evidence that already exists on disk.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "forecasting").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import json

import matplotlib.pyplot as plt
import pandas as pd

from forecasting.tracking import ExperimentTracker
from forecasting.viz import apply_style, CATEGORICAL, INK_SECONDARY
from forecasting.utils import hardware_info

apply_style()
RESULTS_DIR = ROOT / "results"
FIG_DIR = ROOT / "figures"

with open(RESULTS_DIR / "top_squares.json") as f:
    TOP3 = json.load(f)["top3_square_ids"]

MODEL_NAMES = ["SARIMA", "GBM", "LSTM"]

log = ExperimentTracker(RESULTS_DIR / "experiment_log.csv").read_log()
final_log = log[log["phase"] == "final"].copy()
final_log

**Interpretation.** This is every `phase="final"` row `02_experiments.ipynb`
logged: one per (model, square) combination evaluated on the held-out Dec
16-22 test week. Everything below is derived from exactly these 9 rows plus
the predictions/actuals CSVs saved alongside them.

In [ ]:
# Three per-square metric tables (MAE / MAPE / RMSE x 3 models).
per_square_tables = {}
for square_id in TOP3:
    sub = final_log[final_log["square_id"] == square_id].set_index("model")[["mae", "mape", "rmse"]]
    sub = sub.loc[[m for m in MODEL_NAMES if m in sub.index]].round(3)
    per_square_tables[square_id] = sub
    sub.to_csv(RESULTS_DIR / f"metrics_square_{square_id}.csv")
    print(f"--- Square {square_id} ---")
    print(sub, "\n")

**Interpretation.** Read each table for (a) which model has the lowest
MAE/RMSE on that square, and (b) whether MAE and RMSE agree on a winner -
when they don't, it usually means one model has a few unusually large errors
that RMSE's squared penalty punishes more than MAE does. Compare the winner
across all three tables: if the same model wins on every square despite their
different absolute traffic levels (see `01_eda.ipynb`'s distribution figure),
that is evidence the ranking reflects something about the *models*, not an
artifact of one particular square's traffic profile.

In [ ]:
# Nine actual-vs-predicted plots (3 models x 3 squares).
for square_id in TOP3:
    actuals = pd.read_csv(RESULTS_DIR / f"actuals_{square_id}.csv", index_col=0, parse_dates=True)["actual"]
    for model_name in MODEL_NAMES:
        preds = pd.read_csv(RESULTS_DIR / f"predictions_{model_name.lower()}_{square_id}.csv", index_col=0, parse_dates=True)["prediction"]

        fig, ax = plt.subplots(figsize=(9, 3.5))
        ax.plot(actuals.index, actuals.values, color=INK_SECONDARY, label="Actual", linewidth=1.3)
        ax.plot(preds.index, preds.values, color=CATEGORICAL[0], label="Predicted", linewidth=1.1, alpha=0.85)
        ax.set_title(f"{model_name} - square {square_id} - Dec 16-22 one-step-ahead forecast")
        ax.set_xlabel("Date"); ax.set_ylabel("Internet traffic (a.u.)")
        ax.legend(loc="upper right", fontsize=8)
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"forecast_{model_name.lower()}_{square_id}.png")
        plt.show()
        plt.close(fig)

**Interpretation.** Across all nine panels, every model tracks the daily
rise-and-fall shape closely - not surprising given how strong the daily
periodicity found in `01_eda.ipynb` is, which all three input
representations capture one way or another (Fourier terms, calendar
features, or a full-day window). Differences between models show up mainly
at the peaks and in fast local swings rather than in the broad shape: look
for which model's predicted line visibly lags behind or overshoots the
actual line around sharp peaks, and whether that behavior is consistent for
the same model across squares (a systematic property of the model) or
specific to one square (a property of that square's traffic).

In [ ]:
# Training / prediction timing, with hardware noted.
timing_df = pd.read_csv(RESULTS_DIR / "timing.csv")
hw = hardware_info()
print(json.dumps(hw, indent=2))
timing_df

**Interpretation.** All training and prediction times above were measured on
the single machine described by the hardware block - a laptop-class, CPU-only
4-core/11GB machine, not a dedicated training server, which is why every
compute-bounding decision earlier in this project (Fourier terms instead of a
literal seasonal ARIMA, a small LSTM with early stopping, hyperparameter
search on one square only) was made with this hardware specifically in mind.
Compare training time against the accuracy tables above: a model that costs
substantially more to train without a corresponding accuracy advantage is a
real finding worth stating plainly, not a result to explain away.

In [ ]:
# Failure case: the test-week timestamp with the largest mean absolute error
# across all three models, for the top-traffic square.
top_square = TOP3[0]
actuals = pd.read_csv(RESULTS_DIR / f"actuals_{top_square}.csv", index_col=0, parse_dates=True)["actual"]
preds_by_model = {
    m: pd.read_csv(RESULTS_DIR / f"predictions_{m.lower()}_{top_square}.csv", index_col=0, parse_dates=True)["prediction"]
    for m in MODEL_NAMES
}

errs = pd.DataFrame({m: (actuals - preds_by_model[m]).abs() for m in MODEL_NAMES}).dropna()
worst_ts = errs.mean(axis=1).idxmax()
window = slice(worst_ts - pd.Timedelta(hours=6), worst_ts + pd.Timedelta(hours=6))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(actuals.loc[window].index, actuals.loc[window].values, color=INK_SECONDARY, label="Actual", linewidth=1.4)
for i, m in enumerate(MODEL_NAMES):
    p = preds_by_model[m].loc[window]
    ax.plot(p.index, p.values, color=CATEGORICAL[i + 1], label=m, linewidth=1.1, alpha=0.85)
ax.set_title(f"Failure case - square {top_square} - worst mean-error window around {worst_ts}")
ax.set_xlabel("Date"); ax.set_ylabel("Internet traffic (a.u.)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "failure_case.png")
plt.show()

print(f"Worst mean-absolute-error timestamp across all 3 models: {worst_ts}")
errs.loc[window].describe()

**Interpretation.** This is the single hardest moment in the test week for
these three models *combined*, not for any one model in isolation - a
genuinely shared weak point rather than one model's idiosyncratic mistake.
Look at the six-hour window plotted above: does the true series show a sharp,
short-lived move (a local spike or double-peak) that none of the three input
representations - a handful of Fourier harmonics, a small set of discrete
lags, or a one-day sliding window - are built to anticipate ahead of time?
If so, that is a concrete, evidenced limitation worth stating explicitly
rather than only reporting aggregate week-level MAE/MAPE/RMSE, which can look
good on average while still missing moments exactly like this one.